# AquaSentinel: All-Weather Aquaculture (Shrimp) Pond Intelligence

This notebook demonstrates how to monitor coastal aquaculture ponds across the **Bhimavaram / West Godavari cluster (Andhra Pradesh)** using **Sentinel-1 (C-band) + NISAR (L-band) SAR**.

### Core Concepts Demonstrated:
1. **Radar Specular Reflection:** Why open pond water appears pitch-black ($< -20\text{ dB}$) in SAR while dry pond beds appear bright ($-12$ to $-15\text{ dB}$).
2. **All-Weather 3–4 Day Cadence:** Fusing Sentinel-1 and NISAR to defeat coastal monsoon cloud blindness.
3. **State Machine & DOC Tracking:** Stocking date detection, Days of Culture calculation, and **Premature Emergency Drainage (Credit Default) alerts**.

In [ ]:
import sys
from pathlib import Path

src = str(Path("../src").resolve())
if src not in sys.path:
    sys.path.insert(0, src)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sentinel_tool.aquaculture import (
    PondObservation,
    PondState,
    RiskLevel,
    audit_cluster,
    audit_pond_observations,
    generate_benchmark_cluster,
    is_water_backscatter,
)
from sentinel_tool.config import AQUA_AOI_BBOX, AQUA_AOI_NAME

## 1. Study Area: Bhimavaram Aquaculture Belt, Andhra Pradesh
Bhimavaram in West Godavari district is the shrimp capital of India, with over 30,000 active ponds.

In [ ]:
print(f"Target Cluster: {AQUA_AOI_NAME}")
print(f"Bounding Box (W, S, E, N): {AQUA_AOI_BBOX}")

## 2. The Physics: Radar Backscatter of Water vs Dry Earth
In SAR imagery:
- Smooth water acts as a mirror, reflecting radar energy away from the receiver (Backscatter $< -20\text{ dB}$ for S1 VV, $< -22\text{ dB}$ for NISAR HH).
- Dry soil scatters diffuse energy back to the antenna (Backscatter $\approx -12$ to $-15\text{ dB}$).

In [ ]:
db_range = np.linspace(-30, -5, 200)
is_water = [is_water_backscatter(db, "S1", "VV") for db in db_range]

plt.figure(figsize=(10, 3))
plt.plot(db_range, is_water, color="#1f77b4", lw=2.5)
plt.axvline(-19.5, color="red", linestyle="--", label="Water Threshold (-19.5 dB)")
plt.title("Sentinel-1 VV Backscatter Water Classification")
plt.xlabel("Backscatter $\sigma^0$ (dB)")
plt.ylabel("Classification (1 = Water / Stocked, 0 = Dry Bed)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 3. Auditing a Pond Time Series
Let's trace two real-world operational scenarios:
1. **Farmer A (Normal Cycle):** Stocked in May, grown for 88 days, harvested at full maturity.
2. **Farmer B (Emergency Disease Drain):** Stocked in June, hit by WSSV at Day 42, secretly drained and sold to cash buyer.

In [ ]:
# Scenario 1: Normal commercial harvest (DOC 88)
obs_normal = [
    PondObservation("2025-05-01", "S1", "VV", -13.5),   # Dry bed
    PondObservation("2025-05-15", "S1", "VV", -22.4),   # Stocking Day 0
    PondObservation("2025-06-15", "NISAR", "HH", -23.8), # Day 31
    PondObservation("2025-07-15", "S1", "VV", -21.9),   # Day 61
    PondObservation("2025-08-11", "S1", "VV", -13.2),   # Harvested Day 88
]
res_normal = audit_pond_observations("AP-BV-01", obs_normal, farmer_name="K. Suresh Varma", as_of_date="2025-08-15")

# Scenario 2: Premature disease drainage (DOC 42)
obs_alert = [
    PondObservation("2025-06-01", "S1", "VV", -13.2),   # Dry bed
    PondObservation("2025-06-10", "S1", "VV", -22.6),   # Stocking Day 0
    PondObservation("2025-07-05", "NISAR", "HH", -24.1), # Day 25
    PondObservation("2025-07-22", "S1", "VV", -13.6),   # Drained suddenly on Day 42!
]
res_alert = audit_pond_observations("AP-BV-02", obs_alert, farmer_name="V. Rama Rao", credit_amount_inr=350000, as_of_date="2025-07-25")

print("=== POND 1 (Normal Harvest) ===")
print(f"State: {res_normal.current_state.value} | Risk: {res_normal.risk_level.value}")
print(f"Summary: {res_normal.summary}\n")

print("=== POND 2 (Disease Crash Alert) ===")
print(f"State: {res_alert.current_state.value} | Risk: {res_alert.risk_level.value} | At Risk: Rs. {res_alert.credit_at_risk_inr:,.0f}")
print(f"Summary: {res_alert.summary}")

## 4. Visualizing the Backscatter Time-Series Drop & Jump

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4), sharey=True)

# Plot Normal Pond
dates1 = [o.date_str for o in obs_normal]
db1 = [o.backscatter_db for o in obs_normal]
ax1.plot(dates1, db1, marker="o", lw=2, color="#2ca02c", label="Backscatter (dB)")
ax1.axhline(-19.5, color="red", linestyle="--", alpha=0.7, label="Water Threshold")
ax1.set_title("Pond AP-BV-01: Normal Harvest (DOC 88)")
ax1.set_ylabel("SAR Backscatter (dB)")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot Alert Pond
dates2 = [o.date_str for o in obs_alert]
db2 = [o.backscatter_db for o in obs_alert]
ax2.plot(dates2, db2, marker="s", lw=2, color="#d62728", label="Backscatter (dB)")
ax2.axhline(-19.5, color="red", linestyle="--", alpha=0.7, label="Water Threshold")
ax2.set_title("Pond AP-BV-02: Premature Drainage Alert (DOC 42)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Dealership Portfolio Audit (30-Pond Cluster)
Run an automated portfolio-level credit audit over the Bhimavaram dealership cluster.

In [ ]:
ponds = generate_benchmark_cluster(cluster_name="Bhimavaram", count=30, seed=42)
cluster_summary = audit_cluster(ponds, as_of_date="2025-08-15")

print(f"Cluster: {cluster_summary['cluster_name']} (as of {cluster_summary['as_of_date']})")
print(f"Total Ponds:              {cluster_summary['total_ponds']}")
print(f"Stocked / Active:         {cluster_summary['active_stocked_count']} ({cluster_summary['stocking_rate_pct']}%)")
print(f"Premature Drained (Risk): {cluster_summary['premature_drain_count']} ({cluster_summary['premature_drain_rate_pct']}%)")
print(f"Normal Harvested:         {cluster_summary['normal_harvest_count']}")
print(f"Total Credit Portfolio:   Rs. {cluster_summary['total_credit_portfolio_inr']:,.0f}")
print(f"Critical Credit at Risk:  Rs. {cluster_summary['critical_credit_at_risk_inr']:,.0f}")

# Bar chart of portfolio states
states = ["Stocked", "Premature Drain", "Harvested", "Fallow"]
counts = [
    cluster_summary['active_stocked_count'],
    cluster_summary['premature_drain_count'],
    cluster_summary['normal_harvest_count'],
    cluster_summary['fallow_count'],
]
colors = ["#2ca02c", "#d62728", "#1f77b4", "#7f7f7f"]

plt.figure(figsize=(7, 4))
plt.bar(states, counts, color=colors)
plt.title("Bhimavaram Cluster Pond Portfolio Distribution")
plt.ylabel("Number of Ponds")
plt.grid(axis="y", alpha=0.3)
plt.show()